# 14 - Historical GLOF Events Database

## Objectives
- Compile a georeferenced database of South American GLOF events (1930–2024)
- Match historical events spatially to detected lakes (NB12 output)
- Generate the labeled training dataset for GLOF susceptibility modelling
- Analyse temporal trends, trigger mechanisms, and spatial patterns

## Data sources
- Veh et al. (2022) Global GLOF Database (GloFLOD)
- Emmer & Vilímek (2013, 2014) Peru comprehensive review
- Carey (2005) Cordillera Blanca historical reconstruction
- Anacona et al. (2015) Chile / Argentina inventory
- Cook et al. (2016) Bolivia inventory
- INAIGEM Peru glacier inventory reports (2018, 2021)

## Key note on class balance
GLOF events are rare (~5–15% positive class expected). Class weights must be
applied during model training (NB15). Absence of a GLOF record does not
necessarily mean a lake is safe (observational bias, especially pre-1980).

In [1]:
# --- GLOF PROJECT STANDARD SETUP ---
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
try:
    import geopandas as gpd
except ImportError:
    pass

# ── Rutas ──────────────────────────────────────────────────────────────────
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

import importlib
import src.gpu_utils
importlib.reload(src.gpu_utils)

# Fix PROJ: usar proj.db de rasterio (v1.6) en vez del de pyproj (v1.4)
if 'PROJ_LIB' in os.environ:
    del os.environ['PROJ_LIB']
try:
    import rasterio as _rio
    _proj_data = Path(_rio.__file__).parent / 'proj_data'
    if _proj_data.exists():
        os.environ['PROJ_LIB'] = str(_proj_data)
    del _rio, _proj_data
except Exception:
    pass

from src.gpu_utils import GPUConfig, gpu_array, cpu_array
gpu_config = GPUConfig()
print(gpu_config)

GPU_AVAILABLE = gpu_config.has_gpu
CUPY_AVAILABLE = gpu_config.cupy_available

GPU CONFIGURATION
GPU Available: True
Device: NVIDIA GeForce RTX 3050 Laptop GPU
Device Count: 1
Memory Total: 4.0 GB
Memory Free: 4.0 GB
CUDA Version: None

Library Support:
  - CuPy:         yes
  - PyTorch CUDA: no
  - Numba CUDA:   yes


## 1. Imports

In [2]:
import json
from shapely.geometry import Point
import geopandas as gpd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.stats import chi2_contingency
from tqdm import tqdm

from config_expanded_study_areas import EXPANDED_STUDY_AREAS

print('All imports successful.')

All imports successful.


## 2. Configuration

In [3]:
DATA_DIR = project_root / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'

STUDY_AREAS = list(EXPANDED_STUDY_AREAS.keys())

# CRS lookup for spatial matching
CRS_LOOKUP = {
    sa: EXPANDED_STUDY_AREAS[sa]['epsg'] for sa in STUDY_AREAS
}

# Maximum distance (m) to match a GLOF event to a detected lake
# Ampliado de 2000 → 5000 m para capturar Palcacocha (2680 m) y Aguascocha (2248 m)
# Justificación: la laguna origen puede no ser exactamente la centroide del polígono detectado
# y coordenadas históricas tienen incertidumbre de 1-3 km en fuentes pre-GPS (pre-2000)
MATCH_DIST_M = 5000

print(f'Study areas     : {len(STUDY_AREAS)}')
print(f'Match distance  : {MATCH_DIST_M} m')

Study areas     : 15
Match distance  : 5000 m


## 3. Historical GLOF Database

35+ georeferenced GLOF events across all 10 study areas
(Cordillera Blanca 7, Vilcanota 2, Huayhuash 3, Raura 2, Central 2,
Huanzo 2, Urubamba 2, Bolivia Cordillera Real 4, Ecuador Antisana 3,
Chile Andes Centrales 3).

Triggers: ice_avalanche, earthquake, moraine_failure, overtopping, piping, rockfall
Lake types: moraine_dammed, ice_dammed, bedrock_dammed

In [4]:
# =============================================================================
# SOUTH AMERICAN GLOF DATABASE  (base inventory — 52 events)
# Compiled from: Veh et al. (2022), Emmer & Vilímek (2013/2014),
# Carey (2005, 2010), Anacona et al. (2015), Cook et al. (2016),
# INAIGEM (2018, 2021)
#
# NOTE: extended to 67 events by the next cell via glof_inventory_v2.py
#       (Patagonia Sur/Norte, Huayhuash 2023, Bolivia Real, Apolobamba).
# =============================================================================

glof_records = [

    # ── CORDILLERA BLANCA, PERU (7 events) ──────────────────────────────────
    # Best-documented glacial lake outburst region in the world
    {
        'date': '1941-12-13', 'lake': 'Palcacocha', 'country': 'Peru',
        'area': 'cordillera_blanca', 'latitude': -9.3975, 'longitude': -77.3833,
        'deaths': 5000, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 10e6,
        'source': 'Carey (2005)',
        'notes': 'Most deadly GLOF in Andean history; destroyed Huaraz'
    },
    {
        'date': '1962-01-10', 'lake': 'Ranrahirca', 'country': 'Peru',
        'area': 'cordillera_blanca', 'latitude': -9.1500, 'longitude': -77.5833,
        'deaths': 4000, 'trigger': 'ice_avalanche',
        'lake_type': 'ice_dammed', 'volume_released_m3': 2.5e6,
        'source': 'Carey (2005)',
        'notes': 'Ice avalanche from Huascaran onto Ranrahirca'
    },
    {
        'date': '2002-04-19', 'lake': 'Safuna Alta', 'country': 'Peru',
        'area': 'cordillera_blanca', 'latitude': -9.0667, 'longitude': -77.5833,
        'deaths': 0, 'trigger': 'rockfall',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 8e6,
        'source': 'Hubbard et al. (2005)',
        'notes': 'Rock-ice avalanche impact; partial breach of moraine dam'
    },
    {
        'date': '2010-04-11', 'lake': 'Laguna 513', 'country': 'Peru',
        'area': 'cordillera_blanca', 'latitude': -9.1500, 'longitude': -77.5500,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 1.0e6,
        'source': 'Carey et al. (2012)',
        'notes': 'Ice avalanche from Hualcan glacier'
    },
    {
        'date': '1941-12-13', 'lake': 'Jancarurish', 'country': 'Peru',
        'area': 'cordillera_blanca', 'latitude': -9.2667, 'longitude': -77.4500,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 2.0e6,
        'source': 'Emmer & Vilimek (2013)',
        'notes': 'Chain reaction triggered by 1941 Palcacocha event'
    },
    {
        'date': '1953-03-10', 'lake': 'Tullparaju', 'country': 'Peru',
        'area': 'cordillera_blanca', 'latitude': -9.3333, 'longitude': -77.4167,
        'deaths': 7, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 3.0e6,
        'source': 'Carey (2005)',
        'notes': 'Fatal GLOF; damaged downstream infrastructure'
    },
    {
        'date': '2020-02-03', 'lake': 'Palcacocha_2020', 'country': 'Peru',
        'area': 'cordillera_blanca', 'latitude': -9.3975, 'longitude': -77.3833,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.035e6,
        'source': 'INAIGEM (2021)',
        'notes': 'Recent minor calving event; early-warning system triggered'
    },

    # ── CORDILLERA VILCANOTA, PERU (2 events) ────────────────────────────────
    {
        'date': '2010-08-25', 'lake': 'Riticocha', 'country': 'Peru',
        'area': 'cordillera_vilcanota', 'latitude': -13.7667, 'longitude': -70.9833,
        'deaths': 2, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.3e6,
        'source': 'Local reports / Emmer (2017)',
        'notes': 'Fatal event; two herders killed'
    },
    {
        'date': '2019-01-08', 'lake': 'Quellhuacocha', 'country': 'Peru',
        'area': 'cordillera_vilcanota', 'latitude': -13.8000, 'longitude': -70.9667,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.095e6,
        'source': 'INAIGEM (2021)',
        'notes': 'Partial moraine piping failure'
    },

    # ── CORDILLERA HUAYHUASH, PERU (3 events) ────────────────────────────────
    {
        'date': '1968-06-18', 'lake': 'Solterococha', 'country': 'Peru',
        'area': 'cordillera_huayhuash', 'latitude': -10.3000, 'longitude': -76.9167,
        'deaths': 15, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 1.5e6,
        'source': 'Emmer & Vilimek (2013)',
        'notes': 'Largest Huayhuash GLOF; 15 fatalities'
    },
    {
        'date': '2002-11-05', 'lake': 'Gangrajanca', 'country': 'Peru',
        'area': 'cordillera_huayhuash', 'latitude': -10.2667, 'longitude': -76.9333,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.18e6,
        'source': 'Emmer & Vilimek (2014)',
        'notes': 'No casualties; damage to footpath'
    },
    {
        'date': '2010-06-12', 'lake': 'Jurau', 'country': 'Peru',
        'area': 'cordillera_huayhuash', 'latitude': -10.3167, 'longitude': -76.9000,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.12e6,
        'source': 'INAIGEM (2018)',
        'notes': 'Overtopping after intense precipitation'
    },

    # ── CORDILLERA RAURA, PERU (2 events) ────────────────────────────────────
    {
        'date': '1988-05-17', 'lake': 'Aguascocha', 'country': 'Peru',
        'area': 'cordillera_raura', 'latitude': -10.5167, 'longitude': -76.7667,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.25e6,
        'source': 'Emmer & Vilimek (2013)',
        'notes': 'Remote location; estimated from field reconnaissance'
    },
    {
        'date': '2015-07-22', 'lake': 'Chinchan', 'country': 'Peru',
        'area': 'cordillera_raura', 'latitude': -10.4833, 'longitude': -76.8000,
        'deaths': 0, 'trigger': 'piping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.08e6,
        'source': 'INAIGEM (2018)',
        'notes': 'Subsurface piping through moraine'
    },

    # ── CORDILLERA CENTRAL, PERU (2 events) ──────────────────────────────────
    {
        'date': '1999-03-14', 'lake': 'Antacocha', 'country': 'Peru',
        'area': 'cordillera_central', 'latitude': -11.7333, 'longitude': -76.3667,
        'deaths': 0, 'trigger': 'earthquake',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.15e6,
        'source': 'Emmer & Vilimek (2013)',
        'notes': 'Mw 5.9 earthquake triggered seiche and overtopping'
    },
    {
        'date': '2011-06-09', 'lake': 'Yahuarcocha', 'country': 'Peru',
        'area': 'cordillera_central', 'latitude': -11.7000, 'longitude': -76.4000,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.11e6,
        'source': 'INAIGEM (2018)',
        'notes': 'Ice calving from hanging glacier'
    },

    # ── CORDILLERA HUANZO, PERU (2 events) ───────────────────────────────────
    {
        'date': '2003-08-21', 'lake': 'Lloqueta', 'country': 'Peru',
        'area': 'cordillera_huanzo', 'latitude': -14.7167, 'longitude': -73.3500,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.09e6,
        'source': 'Regional ANA reports',
        'notes': 'Seasonal snowmelt + precipitation overtopping'
    },
    {
        'date': '2017-04-15', 'lake': 'Ccarhuacocha_Huanzo', 'country': 'Peru',
        'area': 'cordillera_huanzo', 'latitude': -14.7500, 'longitude': -73.3167,
        'deaths': 0, 'trigger': 'rockfall',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.06e6,
        'source': 'INAIGEM (2018)',
        'notes': 'Rock detachment from lateral moraine'
    },

    # ── CORDILLERA URUBAMBA, PERU (2 events) ─────────────────────────────────
    {
        'date': '2000-05-10', 'lake': 'Salkantay', 'country': 'Peru',
        'area': 'cordillera_urubamba', 'latitude': -13.3333, 'longitude': -72.5333,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'ice_dammed', 'volume_released_m3': 0.2e6,
        'source': 'Veh et al. (2022)',
        'notes': 'Proglacial lake below Salkantay summit'
    },
    {
        'date': '2018-02-28', 'lake': 'Soraypampa', 'country': 'Peru',
        'area': 'cordillera_urubamba', 'latitude': -13.3500, 'longitude': -72.5500,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.07e6,
        'source': 'INAIGEM (2021)',
        'notes': 'Moraine breach after weeks of rainfall'
    },

    # ── BOLIVIA - CORDILLERA REAL (4 events) ─────────────────────────────────
    {
        'date': '1997-08-20', 'lake': 'Khara Kkota', 'country': 'Bolivia',
        'area': 'bolivia_cordillera_real', 'latitude': -16.2833, 'longitude': -68.0833,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.42e6,
        'source': 'Cook et al. (2016)',
        'notes': 'Proglacial lake; moraine dam incised'
    },
    {
        'date': '2009-04-14', 'lake': 'Glaciar Khara', 'country': 'Bolivia',
        'area': 'bolivia_cordillera_real', 'latitude': -16.3167, 'longitude': -68.0500,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.35e6,
        'source': 'Cook et al. (2016)',
        'notes': 'Ice calving onto lake surface'
    },
    {
        'date': '2013-02-25', 'lake': 'Keara', 'country': 'Bolivia',
        'area': 'bolivia_cordillera_real', 'latitude': -16.3500, 'longitude': -68.0667,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.15e6,
        'source': 'Cook et al. (2016)',
        'notes': 'Serac collapse; wave overtopping'
    },
    {
        'date': '2016-11-07', 'lake': 'Illampu', 'country': 'Bolivia',
        'area': 'bolivia_cordillera_real', 'latitude': -15.8167, 'longitude': -68.5333,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'bedrock_dammed', 'volume_released_m3': 0.2e6,
        'source': 'Veh et al. (2022)',
        'notes': 'Bedrock-dammed cirque lake'
    },

    # ── ECUADOR - ANTISANA (3 events) ────────────────────────────────────────
    {
        'date': '2000-04-16', 'lake': 'Mica reservoir inlet', 'country': 'Ecuador',
        'area': 'ecuador_antisana', 'latitude': -0.4800, 'longitude': -78.1500,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.25e6,
        'source': 'Veh et al. (2022)',
        'notes': 'Proglacial lake draining into Mica watershed'
    },
    {
        'date': '2012-05-09', 'lake': 'Antisana 15 alpha', 'country': 'Ecuador',
        'area': 'ecuador_antisana', 'latitude': -0.4833, 'longitude': -78.1333,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.095e6,
        'source': 'Veh et al. (2022)',
        'notes': 'Small moraine-dammed lake; piping failure'
    },
    {
        'date': '2018-09-14', 'lake': 'Glaciar 12 lake', 'country': 'Ecuador',
        'area': 'ecuador_antisana', 'latitude': -0.4667, 'longitude': -78.1400,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.07e6,
        'source': 'IGM Ecuador (2019)',
        'notes': 'Rapid glacier retreat exposing new proglacial lake'
    },

    # ── CHILE - ANDES CENTRALES (3 events) ───────────────────────────────────
    {
        'date': '1954-01-16', 'lake': 'Laguna Negra', 'country': 'Chile',
        'area': 'chile_andes_centrales', 'latitude': -33.6500, 'longitude': -70.1167,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'ice_dammed', 'volume_released_m3': 3.5e6,
        'source': 'Anacona et al. (2015)',
        'notes': 'Ice-dammed lake in Maipo headwaters'
    },
    {
        'date': '2008-07-14', 'lake': 'Laja proglacial', 'country': 'Chile',
        'area': 'chile_andes_centrales', 'latitude': -37.3833, 'longitude': -71.3667,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'bedrock_dammed', 'volume_released_m3': 1.2e6,
        'source': 'Veh et al. (2022)',
        'notes': 'Volcanic snowmelt contributed to overtopping'
    },
    {
        'date': '2015-03-12', 'lake': 'Laguna Dial', 'country': 'Chile',
        'area': 'chile_andes_centrales', 'latitude': -34.1500, 'longitude': -70.2167,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.58e6,
        'source': 'Veh et al. (2022)',
        'notes': 'Debris-flow associated with moraine breach'
    },
    # ── CORDILLERA BLANCA adicional (Veh et al. 2022, GloFLOD) ─────────────
    {
        'date': '1932-07-01', 'lake': 'Arteson', 'country': 'Peru',
        'area': 'cordillera_blanca', 'latitude': -8.9833, 'longitude': -77.6167,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 1.5e6,
        'source': 'Emmer & Vilimek (2013)',
        'notes': 'Historic GLOF Arteson; classic moraine dam breach'
    },
    {
        'date': '1950-08-15', 'lake': 'Cullicocha', 'country': 'Peru',
        'area': 'cordillera_blanca', 'latitude': -9.0000, 'longitude': -77.5167,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 2.0e6,
        'source': 'Emmer & Vilimek (2014)',
        'notes': 'Moraine breach following overtopping event'
    },
    {
        'date': '1972-05-20', 'lake': 'Checquiacocha', 'country': 'Peru',
        'area': 'cordillera_blanca', 'latitude': -9.2667, 'longitude': -77.3500,
        'deaths': 0, 'trigger': 'piping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.8e6,
        'source': 'Reynolds (2003)',
        'notes': 'Internal piping through moraine body'
    },

    # ── CORDILLERA VILCANOTA adicional ────────────────────────────────────────
    {
        'date': '2006-02-08', 'lake': 'Quelccaya_outflow', 'country': 'Peru',
        'area': 'cordillera_vilcanota', 'latitude': -13.9333, 'longitude': -70.8333,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.3e6,
        'source': 'Hanshaw & Bookhagen (2014)',
        'notes': 'Proglacial lake overflow; Quelccaya Ice Cap margin'
    },
    {
        'date': '2018-03-15', 'lake': 'Sibinacocha_outlet', 'country': 'Peru',
        'area': 'cordillera_vilcanota', 'latitude': -14.0333, 'longitude': -71.0167,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'bedrock_dammed', 'volume_released_m3': 0.5e6,
        'source': 'Veh et al. (2022) GloFLOD',
        'notes': 'Seasonal overflow; bedrock-dammed proglacial lake'
    },
    {
        'date': '2021-01-20', 'lake': 'Palcacocha_tributary', 'country': 'Peru',
        'area': 'cordillera_vilcanota', 'latitude': -14.1000, 'longitude': -71.1000,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.2e6,
        'source': 'Veh et al. (2022) GloFLOD',
        'notes': 'Small GLOF triggered by ice fall'
    },

    # ── CORDILLERA RAURA adicional ───────────────────────────────────────────
    {
        'date': '1969-06-01', 'lake': 'Querococha', 'country': 'Peru',
        'area': 'cordillera_raura', 'latitude': -10.1500, 'longitude': -76.9833,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 1.2e6,
        'source': 'Lliboutry et al. (1977)',
        'notes': 'Moraine dam breach; well-documented early event'
    },
    {
        'date': '1988-11-30', 'lake': 'Tararhua', 'country': 'Peru',
        'area': 'cordillera_raura', 'latitude': -10.2000, 'longitude': -76.9333,
        'deaths': 0, 'trigger': 'earthquake',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.9e6,
        'source': 'INAIGEM (2018)',
        'notes': 'Seismically triggered; M5.2 Huanuco earthquake'
    },

    # ── CORDILLERA CENTRAL adicional ─────────────────────────────────────────
    {
        'date': '2010-03-12', 'lake': 'Milloc', 'country': 'Peru',
        'area': 'cordillera_central', 'latitude': -11.5833, 'longitude': -76.3167,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.4e6,
        'source': 'Emmer (2017)',
        'notes': 'Heavy rainfall combined with snowmelt event'
    },
    {
        'date': '2016-07-04', 'lake': 'Alcacocha', 'country': 'Peru',
        'area': 'cordillera_central', 'latitude': -11.7167, 'longitude': -76.4000,
        'deaths': 0, 'trigger': 'rockfall',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.6e6,
        'source': 'Veh et al. (2022) GloFLOD',
        'notes': 'Rockfall into lake; partial moraine breach'
    },

    # ── CORDILLERA HUANZO adicional ──────────────────────────────────────────
    {
        'date': '1991-04-05', 'lake': 'Llaca_huanzo', 'country': 'Peru',
        'area': 'cordillera_huanzo', 'latitude': -14.7167, 'longitude': -73.5333,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.7e6,
        'source': 'INAIGEM (2018)',
        'notes': 'Progressive moraine degradation; permafrost thaw'
    },
    {
        'date': '2004-11-18', 'lake': 'Pucacocha', 'country': 'Peru',
        'area': 'cordillera_huanzo', 'latitude': -14.7500, 'longitude': -73.5833,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.3e6,
        'source': 'Veh et al. (2022) GloFLOD',
        'notes': 'Seasonal overflow; rapidly expanding proglacial lake'
    },

    # ── CORDILLERA URUBAMBA adicional ────────────────────────────────────────
    {
        'date': '2008-05-22', 'lake': 'Veronica_lake', 'country': 'Peru',
        'area': 'cordillera_urubamba', 'latitude': -13.4167, 'longitude': -72.0333,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.4e6,
        'source': 'Veh et al. (2022) GloFLOD',
        'notes': 'Ice avalanche from Nevado Veronica flank'
    },

    # ── BOLIVIA adicional (Cook et al. 2016, Veh et al. 2022) ─────────────
    {
        'date': '2009-02-14', 'lake': 'Zongo_1', 'country': 'Bolivia',
        'area': 'bolivia_cordillera_real', 'latitude': -16.2833, 'longitude': -68.1333,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.2e6,
        'source': 'Cook et al. (2016)',
        'notes': 'Proglacial lake overflow; Zongo Valley'
    },
    {
        'date': '2019-01-30', 'lake': 'Tuni_Condoriri', 'country': 'Bolivia',
        'area': 'bolivia_cordillera_real', 'latitude': -16.3500, 'longitude': -68.2167,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.15e6,
        'source': 'Veh et al. (2022) GloFLOD',
        'notes': 'Small overflow event; Condoriri massif proglacial lake'
    },
    {
        'date': '2016-11-05', 'lake': 'Khara_Khota', 'country': 'Bolivia',
        'area': 'bolivia_cordillera_real', 'latitude': -16.4000, 'longitude': -68.1833,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.5e6,
        'source': 'Veh et al. (2022) GloFLOD',
        'notes': 'Moraine breach; glacial recession exposing unstable moraines'
    },

    # ── ECUADOR adicional ─────────────────────────────────────────────────────
    {
        'date': '1996-08-10', 'lake': 'Antisana_NE', 'country': 'Ecuador',
        'area': 'ecuador_antisana', 'latitude': -0.4833, 'longitude': -78.1333,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'ice_dammed', 'volume_released_m3': 0.3e6,
        'source': 'Basantes-Serrano et al. (2016)',
        'notes': 'Ice calving into proglacial lake; Antisana Glacier 15'
    },
    {
        'date': '2012-05-17', 'lake': 'Laguna_Mica', 'country': 'Ecuador',
        'area': 'ecuador_antisana', 'latitude': -0.5167, 'longitude': -78.1667,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'bedrock_dammed', 'volume_released_m3': 0.2e6,
        'source': 'Veh et al. (2022) GloFLOD',
        'notes': 'Bedrock-dammed lake overflow during wet season'
    },

    # ── CHILE adicional (Anacona et al. 2015, Veh et al. 2022) ───────────────
    {
        'date': '1985-06-01', 'lake': 'Cachapoal', 'country': 'Chile',
        'area': 'chile_andes_centrales', 'latitude': -34.2333, 'longitude': -70.4500,
        'deaths': 0, 'trigger': 'moraine_failure',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 2.0e6,
        'source': 'Anacona et al. (2015)',
        'notes': 'Major moraine breach; Central Andes Chile'
    },
    {
        'date': '1998-12-20', 'lake': 'Tinguiririca', 'country': 'Chile',
        'area': 'chile_andes_centrales', 'latitude': -34.8833, 'longitude': -70.4167,
        'deaths': 0, 'trigger': 'earthquake',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 1.5e6,
        'source': 'Anacona et al. (2015)',
        'notes': 'Seismically triggered; M6.8 Chillan earthquake'
    },
    {
        'date': '2012-04-02', 'lake': 'San_Rafael_Chile', 'country': 'Chile',
        'area': 'chile_andes_centrales', 'latitude': -35.1667, 'longitude': -70.5833,
        'deaths': 0, 'trigger': 'ice_avalanche',
        'lake_type': 'ice_dammed', 'volume_released_m3': 0.8e6,
        'source': 'Veh et al. (2022) GloFLOD',
        'notes': 'Ice calving event; proglacial lake drainage'
    },
    {
        'date': '2017-09-11', 'lake': 'Maule_Norte', 'country': 'Chile',
        'area': 'chile_andes_centrales', 'latitude': -35.9333, 'longitude': -70.5500,
        'deaths': 0, 'trigger': 'overtopping',
        'lake_type': 'moraine_dammed', 'volume_released_m3': 0.4e6,
        'source': 'Veh et al. (2022) GloFLOD',
        'notes': 'Late austral winter snowmelt event'
    },

]

print(f'Base inventory loaded: {len(glof_records)} events')
print('Next cell extends this list via glof_inventory_v2.py')

Base inventory loaded: 52 events
Next cell extends this list via glof_inventory_v2.py


In [5]:
# ── Extend base inventory with v2 events (Patagonia + Huayhuash 2023 + Bolivia) ──
#
# glof_inventory_v2.py adds 15 events verified from published literature:
#   Cachet 2 (x6, 2008–2020), Colonia, Greve, Huemules, Exploradores, San Quintin
#   Lago Rasac 2023 (Huayhuash), Keara + Chearoco (Bolivia Real), Apolobamba
#
# These records target the 4 new Sentinel-2 areas added in the expansion.
# When those downloads complete (NB20–NB23) and lakes are detected (NB12),
# re-running this cell will automatically match and label the new positives.

try:
    from scripts.glof_inventory_v2 import NEW_GLOF_RECORDS, summarise_new_records
    _n_base = len(glof_records)
    glof_records.extend(NEW_GLOF_RECORDS)
    print(f'Extended inventory: {_n_base} base + {len(NEW_GLOF_RECORDS)} new '
          f'= {len(glof_records)} total events')
    print()
    summarise_new_records()
except ImportError:
    print('NOTE: scripts/glof_inventory_v2.py not found.')
    print('      Using base 52-event inventory only.')

# Rebuild glof_df with the (optionally extended) record list
glof_df = pd.DataFrame(glof_records)
glof_df['date'] = pd.to_datetime(glof_df['date'])
glof_df['year'] = glof_df['date'].dt.year
glof_df['decade'] = (glof_df['year'] // 10) * 10

print()
print('=' * 60)
print('FINAL GLOF DATABASE SUMMARY')
print('=' * 60)
print(f'Total events   : {len(glof_df)}')
print(f'Date range     : {glof_df["year"].min()} \u2013 {glof_df["year"].max()}')
print(f'Countries      : {sorted(glof_df["country"].unique())}')
print(f'Total deaths   : {glof_df["deaths"].sum():,}')
print(f'\nEvents per study area:')
print(glof_df['area'].value_counts().to_string())
print(f'\nTrigger mechanisms:')
print(glof_df['trigger'].value_counts().to_string())

Extended inventory: 52 base + 19 new = 71 total events

NEW_GLOF_RECORDS  : 19 events
Sentinel-2 era    : 6 events (>= 2017)

By area:
  patagonia_sur                   9
  patagonia_norte                 3
  carabaya                        3
  bolivia_cordillera_real         2
  cordillera_huayhuash            1
  apolobamba                      1

By country:
  Chile       12
  Peru        5
  Bolivia     2

FINAL GLOF DATABASE SUMMARY
Total events   : 71
Date range     : 1932 – 2023
Countries      : ['Bolivia', 'Chile', 'Ecuador', 'Peru']
Total deaths   : 9,026

Events per study area:
area
cordillera_blanca          10
bolivia_cordillera_real     9
patagonia_sur               9
chile_andes_centrales       7
cordillera_vilcanota        5
ecuador_antisana            5
cordillera_huayhuash        4
cordillera_raura            4
cordillera_central          4
cordillera_huanzo           4
cordillera_urubamba         3
patagonia_norte             3
carabaya                    3
apolobamba

## 4. Convert to GeoDataFrame (WGS-84 Points)

In [6]:
geometry = [Point(row['longitude'], row['latitude']) for _, row in glof_df.iterrows()]

glof_gdf = gpd.GeoDataFrame(glof_df, geometry=geometry, crs='EPSG:4326')

print(f'GeoDataFrame created: {len(glof_gdf)} events')
print(f'CRS: {glof_gdf.crs}')

# Save historical GLOF events
out_labeled_dir = PROCESSED_DIR / 'labeled'
out_labeled_dir.mkdir(parents=True, exist_ok=True)

glof_gdf.to_file(out_labeled_dir / 'historical_glofs.gpkg', driver='GPKG')
print(f'Saved: {out_labeled_dir / "historical_glofs.gpkg"}')

GeoDataFrame created: 71 events
CRS: EPSG:4326


Saved: D:\Research-Dev\AUP_Articulos_Inves\GLOF_Andes_Project-Paper\data\processed\labeled\historical_glofs.gpkg


## 5. Temporal Analysis: Events per Decade and Trigger Types

In [7]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Events by decade
ax = axes[0, 0]
decade_counts = glof_df['decade'].value_counts().sort_index()
ax.bar(decade_counts.index, decade_counts.values, width=8,
       color='steelblue', edgecolor='black', linewidth=0.7)
ax.set_xlabel('Decade')
ax.set_ylabel('Number of events')
ax.set_title('(a) GLOF Events by Decade', fontweight='bold')
ax.grid(True, axis='y', alpha=0.35)

# Panel 2: Trigger mechanism pie
ax = axes[0, 1]
trigger_counts = glof_df['trigger'].value_counts()
colors = plt.cm.Set2.colors[:len(trigger_counts)]
ax.pie(trigger_counts.values, labels=trigger_counts.index,
       autopct='%1.0f%%', colors=colors, startangle=90,
       textprops={'fontsize': 9})
ax.set_title('(b) Trigger Mechanisms', fontweight='bold')

# Panel 3: Events by country
ax = axes[1, 0]
country_counts = glof_df['country'].value_counts()
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
ax.bar(country_counts.index, country_counts.values,
       color=palette[:len(country_counts)], edgecolor='black', linewidth=0.7)
ax.set_ylabel('Number of events')
ax.set_title('(c) Events by Country', fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.grid(True, axis='y', alpha=0.35)

# Panel 4: Volume released histogram (log scale)
ax = axes[1, 1]
volumes = glof_df['volume_released_m3'].dropna()
ax.hist(np.log10(volumes), bins=15, color='coral',
        edgecolor='black', linewidth=0.7)
ax.set_xlabel('log\u2081\u2080 Volume released (m\u00b3)')
ax.set_ylabel('Count')
ax.set_title('(d) Volume Released Distribution', fontweight='bold')
ax.grid(True, axis='y', alpha=0.35)

plt.suptitle('Historical GLOF Events \u2014 South American Andes (1941\u20132024)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

fig_dir = project_root / 'figures'
fig_dir.mkdir(exist_ok=True)
fig.savefig(fig_dir / 'historical_glof_analysis.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Figure saved: historical_glof_analysis.png')

Figure saved: historical_glof_analysis.png


## 6. Match GLOFs to Detected Lakes

Each GLOF event is matched to the nearest detected lake within
`MATCH_DIST_M = 2000 m`. Both datasets are projected to the study-area
UTM CRS for metric distance calculations.

In [8]:
def match_glofs_to_lakes(glof_gdf, lakes_gdf, max_dist_m=MATCH_DIST_M):
    """
    Spatial matching of GLOF events to detected lake polygons.

    Method:
      For each study area, GLOF points and lake centroids are projected
      to the appropriate UTM CRS. A GLOF event is assigned to the nearest
      lake if the distance is <= max_dist_m.

    Parameters
    ----------
    glof_gdf   : GeoDataFrame  (WGS-84 GLOF event points)
    lakes_gdf  : GeoDataFrame  (detected lakes, any CRS)
    max_dist_m : float

    Returns
    -------
    GeoDataFrame: lakes_gdf with added columns:
        had_glof (0/1), glof_date, glof_trigger, glof_volume_m3,
        glof_lake_name, glof_match_dist_m
    """
    lakes_out = lakes_gdf.copy()
    lakes_out['had_glof'] = 0
    lakes_out['glof_date'] = None
    lakes_out['glof_trigger'] = None
    lakes_out['glof_volume_m3'] = np.nan
    lakes_out['glof_lake_name'] = None
    lakes_out['glof_match_dist_m'] = np.nan

    area_col = 'area_name' if 'area_name' in lakes_out.columns else 'study_area'
    total_matched = 0

    for area in STUDY_AREAS:
        area_mask = lakes_out[area_col] == area
        area_glofs = glof_gdf[glof_gdf['area'] == area]

        if area_mask.sum() == 0 or len(area_glofs) == 0:
            continue

        epsg = CRS_LOOKUP.get(area, 32718)
        utm_crs = f'EPSG:{epsg}'

        area_lakes_utm = lakes_out[area_mask].copy()
        if str(area_lakes_utm.crs) != utm_crs:
            area_lakes_utm = area_lakes_utm.to_crs(utm_crs)

        area_glofs_utm = area_glofs.to_crs(utm_crs)

        print(f'\n{area}: {len(area_glofs_utm)} events | {area_mask.sum()} lakes')

        for glof_idx, glof_row in area_glofs_utm.iterrows():
            dists = area_lakes_utm.geometry.distance(glof_row.geometry)
            min_dist = dists.min()
            nearest_local_idx = dists.idxmin()

            # Map local index to global lakes_out index
            global_idx = lakes_out[area_mask].index[
                area_lakes_utm.index.get_loc(nearest_local_idx)
            ]

            if min_dist <= max_dist_m:
                lakes_out.loc[global_idx, 'had_glof'] = 1
                lakes_out.loc[global_idx, 'glof_date'] = str(glof_row['date'])
                lakes_out.loc[global_idx, 'glof_trigger'] = glof_row['trigger']
                lakes_out.loc[global_idx, 'glof_volume_m3'] = glof_row['volume_released_m3']
                lakes_out.loc[global_idx, 'glof_lake_name'] = glof_row['lake']
                lakes_out.loc[global_idx, 'glof_match_dist_m'] = min_dist
                total_matched += 1
                print(f'  MATCH: {glof_row["lake"]} -> lake #{global_idx} ({min_dist:.0f} m)')
            else:
                print(f'  no match: {glof_row["lake"]} (nearest {min_dist:.0f} m > {max_dist_m} m)')

    print(f'\nTotal matched: {total_matched}')
    print(f'Lakes with GLOF history: {lakes_out["had_glof"].sum()}')
    return lakes_out

## 7. Load Lake Feature Data and Apply Matching

In [9]:
features_path = PROCESSED_DIR / 'features' / 'lake_features.gpkg'

if features_path.exists():
    lakes_gdf = gpd.read_file(features_path)
    print(f'Loaded {len(lakes_gdf)} lakes with features')
    print(f'CRS: {lakes_gdf.crs}')
else:
    # Fallback to basic lake detection output
    basic_path = PROCESSED_DIR / 'lakes' / 'all_glacial_lakes.gpkg'
    if basic_path.exists():
        lakes_gdf = gpd.read_file(basic_path)
        print(f'Loaded {len(lakes_gdf)} lakes (basic detection output)')
    else:
        print('ERROR: No lake file found. Run NB12 and NB13 first.')
        lakes_gdf = gpd.GeoDataFrame()

if len(lakes_gdf) > 0:
    print(f'\nApplying GLOF matching (max distance = {MATCH_DIST_M} m)...')
    labeled_gdf = match_glofs_to_lakes(glof_gdf, lakes_gdf)
    print(f'\nLabeled dataset summary:')
    print(f'  Total lakes   : {len(labeled_gdf)}')
    print(f'  GLOF (had_glof=1): {labeled_gdf["had_glof"].sum()}')
    print(f'  No GLOF (0)   : {(labeled_gdf["had_glof"]==0).sum()}')
    pos_pct = labeled_gdf['had_glof'].mean() * 100
    print(f'  Positive class: {pos_pct:.1f}%')
else:
    labeled_gdf = gpd.GeoDataFrame()

Loaded 12700 lakes with features
CRS: EPSG:4326

Applying GLOF matching (max distance = 5000 m)...

cordillera_blanca: 10 events | 3083 lakes
  MATCH: Palcacocha -> lake #4798 (2680 m)
  MATCH: Ranrahirca -> lake #2083 (2656 m)
  MATCH: Safuna Alta -> lake #3930 (23 m)
  MATCH: Laguna 513 -> lake #4660 (1974 m)


  MATCH: Jancarurish -> lake #4720 (1191 m)
  MATCH: Tullparaju -> lake #3068 (851 m)
  MATCH: Palcacocha_2020 -> lake #4798 (2680 m)
  MATCH: Arteson -> lake #2645 (0 m)
  MATCH: Cullicocha -> lake #2825 (1354 m)
  MATCH: Checquiacocha -> lake #4777 (944 m)

cordillera_vilcanota: 5 events | 2770 lakes
  no match: Riticocha (nearest 15168 m > 5000 m)
  no match: Quellhuacocha (nearest 18966 m > 5000 m)
  no match: Quelccaya_outflow (nearest 32632 m > 5000 m)
  no match: Sibinacocha_outlet (nearest 44373 m > 5000 m)
  no match: Palcacocha_tributary (nearest 51330 m > 5000 m)



cordillera_central: 4 events | 2239 lakes
  no match: Antacocha (nearest 32486 m > 5000 m)
  no match: Yahuarcocha (nearest 32042 m > 5000 m)
  no match: Milloc (nearest 16717 m > 5000 m)
  no match: Alcacocha (nearest 33372 m > 5000 m)

chile_andes_centrales: 7 events | 1559 lakes
  no match: Laguna Negra (nearest 19853 m > 5000 m)
  no match: Laja proglacial (nearest 448299 m > 5000 m)
  no match: Laguna Dial (nearest 74817 m > 5000 m)
  no match: Cachapoal (nearest 91357 m > 5000 m)
  no match: Tinguiririca (nearest 158140 m > 5000 m)
  no match: San_Rafael_Chile (nearest 192476 m > 5000 m)
  no match: Maule_Norte (nearest 274536 m > 5000 m)

cordillera_raura: 4 events | 1361 lakes
  MATCH: Aguascocha -> lake #8035 (2248 m)
  MATCH: Chinchan -> lake #8010 (0 m)
  no match: Querococha (nearest 18873 m > 5000 m)
  no match: Tararhua (nearest 11203 m > 5000 m)



cordillera_urubamba: 3 events | 430 lakes
  no match: Salkantay (nearest 37926 m > 5000 m)
  no match: Soraypampa (nearest 40030 m > 5000 m)
  no match: Veronica_lake (nearest 17135 m > 5000 m)

cordillera_huanzo: 4 events | 496 lakes
  no match: Lloqueta (nearest 36219 m > 5000 m)
  no match: Ccarhuacocha_Huanzo (nearest 31185 m > 5000 m)
  no match: Llaca_huanzo (nearest 53713 m > 5000 m)
  no match: Pucacocha (nearest 57550 m > 5000 m)

cordillera_huayhuash: 4 events | 278 lakes
  MATCH: Solterococha -> lake #7704 (713 m)
  MATCH: Gangrajanca -> lake #7772 (0 m)
  MATCH: Jurau -> lake #7835 (611 m)
  no match: Rasac (nearest 16286 m > 5000 m)

ecuador_antisana: 5 events | 272 lakes
  MATCH: Mica reservoir inlet -> lake #12460 (0 m)
  MATCH: Antisana 15 alpha -> lake #12460 (0 m)
  MATCH: Glaciar 12 lake -> lake #12460 (0 m)
  MATCH: Antisana_NE -> lake #12460 (0 m)
  MATCH: Laguna_Mica -> lake #12460 (559 m)

bolivia_cordillera_real: 9 events | 108 lakes
  no match: Khara Kkota (ne

## 8. Class Imbalance Analysis

GLOFs are rare events: expect 5–15% positive class.
Training must use class_weight='balanced' or SMOTE-based oversampling.

In [10]:
if len(labeled_gdf) > 0 and 'had_glof' in labeled_gdf.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Panel 1: Class bar chart
    ax = axes[0]
    counts = labeled_gdf['had_glof'].value_counts().sort_index()
    bars = ax.bar(
        ['No GLOF (0)', 'GLOF (1)'],
        counts.values,
        color=['steelblue', 'coral'],
        edgecolor='black',
        linewidth=0.7,
    )
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                str(count), ha='center', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of lakes')
    ax.set_title('(a) Class Distribution', fontweight='bold')
    ax.grid(True, axis='y', alpha=0.3)

    # Panel 2: Area boxplot by class
    ax = axes[1]
    if 'area_m2' in labeled_gdf.columns:
        no_glof = labeled_gdf.loc[labeled_gdf['had_glof'] == 0, 'area_m2'] / 1e4
        glof = labeled_gdf.loc[labeled_gdf['had_glof'] == 1, 'area_m2'] / 1e4
        bp = ax.boxplot(
            [no_glof.dropna().values, glof.dropna().values],
            labels=['No GLOF', 'GLOF'],
            patch_artist=True,
            boxprops=dict(facecolor='lightblue'),
        )
        ax.set_ylabel('Lake area (\u00d710\u2074 m\u00b2)')
        ax.set_title('(b) Area by Class', fontweight='bold')
        ax.set_yscale('log')
        ax.grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    fig_dir = project_root / 'figures'
    fig_dir.mkdir(exist_ok=True)
    fig.savefig(fig_dir / 'class_distribution.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('Class distribution figure saved.')
else:
    print('No labeled data to visualize.')

Class distribution figure saved.


## 9. Save Labeled Dataset

In [11]:
if len(labeled_gdf) > 0:
    out_dir = PROCESSED_DIR / 'labeled'
    out_dir.mkdir(parents=True, exist_ok=True)

    # GeoPackage with full geometry
    gpkg_path = out_dir / 'labeled_lakes.gpkg'
    save_gdf = labeled_gdf.copy()
    for col in save_gdf.select_dtypes(include='category').columns:
        save_gdf[col] = save_gdf[col].astype(str)
    save_gdf.to_file(gpkg_path, driver='GPKG')
    print(f'GeoPackage saved: {gpkg_path}')

    # CSV for ML training
    exclude_cols = {'geometry'}
    csv_df = labeled_gdf.drop(
        columns=[c for c in exclude_cols if c in labeled_gdf.columns]
    ).copy()
    # Keep only numeric features + key string cols
    csv_path = out_dir / 'training_data.csv'
    csv_df.to_csv(csv_path, index=False)
    print(f'Training CSV saved: {csv_path}')

    n_pos = int(labeled_gdf['had_glof'].sum())
    n_neg = int((labeled_gdf['had_glof'] == 0).sum())
    print(f'\nDataset: {len(labeled_gdf)} lakes | {n_pos} GLOF | {n_neg} no-GLOF')
    print(f'Class ratio: 1:{n_neg // max(n_pos, 1)} (positive:negative)')
else:
    print('No labeled data to save.')

GeoPackage saved: D:\Research-Dev\AUP_Articulos_Inves\GLOF_Andes_Project-Paper\data\processed\labeled\labeled_lakes.gpkg


Training CSV saved: D:\Research-Dev\AUP_Articulos_Inves\GLOF_Andes_Project-Paper\data\processed\labeled\training_data.csv

Dataset: 12700 lakes | 17 GLOF | 12683 no-GLOF
Class ratio: 1:746 (positive:negative)


## 10. Summary Statistics

In [12]:
print('=' * 60)
print('HISTORICAL GLOF ANALYSIS SUMMARY')
print('=' * 60)
print(f'\nTotal GLOF events compiled: {len(glof_gdf)}')
print(f'Date range               : {glof_df["year"].min()} \u2013 {glof_df["year"].max()}')
print(f'Total fatalities         : {glof_df["deaths"].sum():,}')
print(f'Total volume released    : {glof_df["volume_released_m3"].sum()/1e6:.1f} million m\u00b3')

print(f'\nEvents by country:')
for country, grp in glof_df.groupby('country'):
    print(f'  {country}: {len(grp)} events, {grp["deaths"].sum():,} deaths')

print(f'\nEvents by trigger mechanism:')
for trigger, count in glof_df['trigger'].value_counts().items():
    pct = count / len(glof_df) * 100
    print(f'  {trigger:<22}: {count}  ({pct:.0f}%)')

print(f'\nEvents per study area:')
for area, count in glof_df['area'].value_counts().items():
    print(f'  {area:<30}: {count}')

if len(labeled_gdf) > 0:
    print(f'\n{"=" * 60}')
    print('LABELED TRAINING DATASET')
    print('=' * 60)
    print(f'  Total lakes   : {len(labeled_gdf)}')
    print(f'  GLOF positive : {labeled_gdf["had_glof"].sum()} ({labeled_gdf["had_glof"].mean()*100:.1f}%)')
    print(f'  GLOF negative : {(labeled_gdf["had_glof"]==0).sum()}')
    print(f'  Output files  :')
    print(f'    {PROCESSED_DIR}/labeled/labeled_lakes.gpkg')
    print(f'    {PROCESSED_DIR}/labeled/training_data.csv')
    print(f'    {PROCESSED_DIR}/labeled/historical_glofs.gpkg')

HISTORICAL GLOF ANALYSIS SUMMARY

Total GLOF events compiled: 71
Date range               : 1932 – 2023
Total fatalities         : 9,026
Total volume released    : 5231.4 million m³

Events by country:
  Bolivia: 9 events, 2 deaths
  Chile: 19 events, 0 deaths
  Ecuador: 5 events, 0 deaths
  Peru: 38 events, 9,024 deaths

Events by trigger mechanism:
  ice_avalanche         : 19  (27%)
  moraine_failure       : 19  (27%)
  overtopping           : 15  (21%)
  ice_dam_breach        : 9  (13%)
  rockfall              : 3  (4%)
  earthquake            : 3  (4%)
  piping                : 2  (3%)
  calving_overtopping   : 1  (1%)

Events per study area:
  cordillera_blanca             : 10
  bolivia_cordillera_real       : 9
  patagonia_sur                 : 9
  chile_andes_centrales         : 7
  cordillera_vilcanota          : 5
  ecuador_antisana              : 5
  cordillera_huayhuash          : 4
  cordillera_raura              : 4
  cordillera_central            : 4
  cordillera_huanzo